# Monthly revenue forecast and scenario dashboard
Run all cells in order. Forecast **6 periods after each selection's latest recorded period**, including the newest partial period in training as requested.

Eligibility uses the latest **completed July–June fiscal year in the extract**: at least 7 distinct recorded months (monthly), or 3 distinct recorded quarters (quarterly). Earlier FYs may have gaps. Transactions are not counted as separate observations.

Dates define the monthly/quarterly timeline; stored accounting FY discrepancies are audited. GL 42, 4860 and 4870 are excluded. All revenue types are included. Missing periods are not zero-filled: fitting uses the latest contiguous block. A recorded zero counts as coverage. Coverage does not prove a period is complete.

The latest partial period is used **as recorded, without prorating**; this can lower the forecast. A stale GL forecasts from its own latest observation, which may be in the past. Inspect the origin and status before use.

## Monthly data refresh
Replace `data/dawn_revenue_accounts_combined_history.csv` with the updated full history, then restart the kernel and **Run All**. Dates, coverage, BA/GL selections, forecasts, validation and dashboard outputs recalculate automatically. Corrections invalidate the cached fits even if the latest date is unchanged. Reload the HTML dashboard after completion. See the project `README.md` for all cutoff rules and the batch refresh command.

## 1. Settings and imports
The shared module contains commented model functions. Fixed seeds make simulation and Random Forest reproducible. Change settings here and rerun all cells.

In [1]:
from pathlib import Path
import sys, importlib
import pandas as pd
from IPython.display import display, IFrame, FileLink
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'script/period_forecast_core.py').exists())
sys.path.insert(0, str(ROOT / 'script'))
import period_forecast_core as core
importlib.reload(core)
FREQUENCY = 'monthly'
HORIZON = 6
LOOKBACK = 36
SEED = 2026
SOURCE = ROOT / 'data/dawn_revenue_accounts_combined_history.csv'
OUTPUT = ROOT / 'result' / f'{FREQUENCY}_forecast'
OUTPUT.mkdir(parents=True, exist_ok=True)
core.FORECAST_HORIZON = HORIZON
core.LOOKBACK = LOOKBACK
core.SEED = SEED

## 2. Read transactions and audit the calendar
The latest data date determines the completed FY used for eligibility, not a fixed FY2026 cutoff. Newer records remain in the training data.

In [2]:
data, settings, audit = core.prepare_data(SOURCE, FREQUENCY, HORIZON, LOOKBACK)
display(pd.DataFrame([audit]))
print('Coverage FY:', settings['coverage_fy'], '| Data through:', settings['data_through'])

Coverage FY: 2026 | Data through: 2026-10-22


,included_rows,excluded_rows,accounting_fy_date_mismatches,date_basis,latest_date,coverage_fy
0,329413,6228,16506,Transaction Date; July-June FY,2026-10-22,2026


## 3. Coverage by BA and GL
Each BA/GL combination and aggregate is tested separately. The CSV includes excluded selections and reasons; All GLs is a model of aggregate revenue.

In [3]:
selections, coverage = core.make_selections(data, settings)
coverage.to_csv(OUTPUT / 'coverage.csv', index=False)
display(coverage)
print(coverage['eligible'].value_counts())

eligible
False    287
True     138
Name: count, dtype: int64


,ba,gl,description,coverage_fy,coverage_count,eligible,coverage_note,latest_partial
0,ALL,ALL,All included GL codes,2026,12,True,Coverage FY2026: 12 recorded periods; requires...,True
1,9010,ALL,All included GL codes,2026,11,True,Coverage FY2026: 11 recorded periods; requires...,True
2,9010,3102,ATHLETIC COMM GATE/TV FEES,2026,11,True,Coverage FY2026: 11 recorded periods; requires...,True
3,9010,3103,ATHLETIC COMM LICENSES/FINES,2026,11,True,Coverage FY2026: 11 recorded periods; requires...,True
4,9010,3271,MISC FINES/FORFEITURES,2026,10,True,Coverage FY2026: 10 recorded periods; requires...,True
...,...,...,...,...,...,...,...,...
420,ALL,4864,TRANSFER FROM ENTERPRISE FUND,2026,2,False,Coverage FY2026: 2 recorded periods; requires ...,False
421,ALL,4865,TRANSFER FROM INT SERV FUND,2026,1,False,Coverage FY2026: 1 recorded periods; requires ...,False
422,ALL,4866,TRANSFER FROM FIDUCIARY FUND,2026,0,False,Coverage FY2026: 0 recorded periods; requires ...,False
423,ALL,4867,TRANSFER FROM UNIVERSITY FUNDS,2026,0,False,Coverage FY2026: 0 recorded periods; requires ...,False


## 4. Fit models and perform chronological validation
Models: naive last observation, Linear trend, damped Holt, Monte Carlo resampled recent dollar changes, and recursive Random Forest with two lags and time. These retain the previous methodology and do not explicitly model seasonality. Up to three rolling origins predict the full horizon; short histories use a limited one-period holdout. Validation targets exclude the extract’s partial last period. Model selection is descriptive, not independent validation. MC P10/P90 are simulation quantiles, not guaranteed confidence bounds. Negative training revenue remains signed.

In [4]:
report = core.run_models(selections, settings, OUTPUT)
print('Finished', len(report['series']), 'selections')

monthly: 25/425
monthly: 50/425
monthly: 75/425
monthly: 100/425
monthly: 125/425
monthly: 150/425
monthly: 175/425
monthly: 200/425
monthly: 225/425
monthly: 250/425
monthly: 275/425
monthly: 300/425
monthly: 325/425
monthly: 350/425
monthly: 375/425
monthly: 400/425
monthly: 425/425
Finished 425 selections


## 5. Forecast values and CSV export
Each model has one row per future period. Missing predictions remain blank. Origin, partial-period flag, coverage and MC bounds accompany the values.

In [5]:
forecasts, validation, backtests = core.export_tables(report, OUTPUT)
display(forecasts.head(30))
display(FileLink(str(OUTPUT / 'forecast_values.csv')))

,BA,GL,origin,coverage_fy,coverage_count,eligible,latest_period_partial,status,model,period,horizon,forecast,MC_P10,MC_P90
0,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Naive baseline,Nov 2026,1,3.360375e+08,NaN,NaN
1,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Naive baseline,Dec 2026,2,3.360375e+08,NaN,NaN
2,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Naive baseline,Jan 2027,3,3.360375e+08,NaN,NaN
3,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Naive baseline,Feb 2027,4,3.360375e+08,NaN,NaN
4,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Naive baseline,Mar 2027,5,3.360375e+08,NaN,NaN
5,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Naive baseline,Apr 2027,6,3.360375e+08,NaN,NaN
6,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Linear,Nov 2026,1,5.535020e+08,NaN,NaN
7,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Linear,Dec 2026,2,5.554480e+08,NaN,NaN
8,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Linear,Jan 2027,3,5.573940e+08,NaN,NaN
9,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Linear,Feb 2027,4,5.593400e+08,NaN,NaN


C:\Users\Syed.Uddin\OneDrive - State of Nevada\DAWN_DATA\dawn_aggregate\result\monthly_forecast\forecast_values.csv

## 6. Validation table and verdict CSV
MAE and RMSE are dollar errors. WAPE uses absolute actual revenue; skill compares matched cases to naive. At least two full-horizon origins are needed for a strong comparative verdict. Limited validation is explicitly labeled.

In [6]:
display(validation.head(25))
display(FileLink(str(OUTPUT / 'model_validation.csv')))
print('Detailed backtest predictions:', OUTPUT / 'backtest_predictions.csv')

Detailed backtest predictions: C:\Users\Syed.Uddin\OneDrive - State of Nevada\DAWN_DATA\dawn_aggregate\result\monthly_forecast\backtest_predictions.csv


,BA,GL,origin,coverage_fy,coverage_count,eligible,latest_period_partial,status,model,cases,origins,mae,rmse,wape,skill,verdict,comparable
0,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Naive baseline,12,2,3.069105e+08,4.000200e+08,52.727501,0.000000,Baseline benchmark,True
1,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Linear,12,2,2.765577e+08,3.388655e+08,47.512855,9.889803,Improves on baseline,True
2,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Holt,12,2,2.700862e+08,3.423455e+08,46.401042,11.998404,Improves on baseline,True
3,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,Monte Carlo,12,2,2.896090e+08,3.488362e+08,49.755086,5.637314,Improves on baseline,True
4,ALL,ALL,Oct 2026,2026,12,True,True,Coverage FY2026: 12 recorded periods; requires...,ML (Random forest),12,2,2.420657e+08,3.401267e+08,41.587100,21.128255,Improves on baseline,True
5,9010,ALL,Oct 2026,2026,11,True,True,Coverage FY2026: 11 recorded periods; requires...,Naive baseline,12,2,4.578217e+05,5.953235e+05,100.322688,0.000000,Baseline benchmark,True
6,9010,ALL,Oct 2026,2026,11,True,True,Coverage FY2026: 11 recorded periods; requires...,Linear,12,2,4.563492e+05,6.752686e+05,100.000000,0.321650,Improves on baseline,True
7,9010,ALL,Oct 2026,2026,11,True,True,Coverage FY2026: 11 recorded periods; requires...,Holt,12,2,6.460692e+05,8.304411e+05,141.573438,-41.118067,Does not improve on baseline,True
8,9010,ALL,Oct 2026,2026,11,True,True,Coverage FY2026: 11 recorded periods; requires...,Monte Carlo,12,2,3.725491e+05,5.428181e+05,81.636861,18.625725,Improves on baseline,True
9,9010,ALL,Oct 2026,2026,11,True,True,Coverage FY2026: 11 recorded periods; requires...,ML (Random forest),12,2,3.911165e+05,5.111009e+05,85.705530,14.570142,Improves on baseline,True


C:\Users\Syed.Uddin\OneDrive - State of Nevada\DAWN_DATA\dawn_aggregate\result\monthly_forecast\model_validation.csv

## 7. Interactive historical versus forecast dashboard
Select BA, GL, display periods and multiple model projections. Scenario starts at 0%; uniform adjustment applies the same percentage to each future period. Cards show base, adjusted and difference. The table includes two recent actual periods and period-over-period changes. Notes are saved locally in this browser. The standalone HTML works offline; rerun the notebook when source data changes.

In [7]:
dashboard = core.write_dashboard(report, ROOT, OUTPUT)
display(FileLink(str(dashboard)))
display(IFrame(src='../result/' + FREQUENCY + '_forecast/' + dashboard.name, width='100%', height=1100))

C:\Users\Syed.Uddin\OneDrive - State of Nevada\DAWN_DATA\dawn_aggregate\result\monthly_forecast\monthly_revenue_dashboard.html